# Customer retention and cohort analysis
Synthetic mobile banking activity; all figures are illustrative.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
base=Path('../data') if Path('../data/users.csv').exists() else Path('data')
users=pd.read_csv(base/'users.csv',parse_dates=['signup_month'])
activity=pd.read_csv(base/'monthly_activity.csv',parse_dates=['activity_month'])
joined=activity.merge(users,on='user_id',validate='many_to_one')
joined['cohort']=joined.signup_month.dt.to_period('M').astype(str)
joined['age_months']=(joined.activity_month.dt.year-joined.signup_month.dt.year)*12+joined.activity_month.dt.month-joined.signup_month.dt.month
cohort_sizes=users.groupby(users.signup_month.dt.to_period('M')).user_id.nunique()
active=joined.groupby(['cohort','age_months']).user_id.nunique().unstack(fill_value=0)
retention=active.div(cohort_sizes.rename(index=str),axis=0)
print('Month 1 retention by cohort:')
print(retention.get(1,pd.Series(dtype=float)).round(3))
plt.figure(figsize=(11,5));sns.heatmap(retention,annot=True,fmt='.0%',cmap='Blues',vmin=0,vmax=1);plt.title('Monthly cohort retention');plt.xlabel('Months since signup');plt.ylabel('Signup cohort');plt.tight_layout();plt.show()
print('Average transaction volume by acquisition channel:')
print(joined.groupby('channel').volume_rub.mean().round(2))


## Interpretation
Look for cohort differences in M1 retention and compare acquisition channels. The generating process includes a referral uplift, so these differences are simulated, not causal evidence.